# Limpieza de Datos de Establecimientos Educativos (Diversificado)
**Integrantes:** 
- Sofía García
- Julio García Salas
- Joaquin Campos 

## Paso 2: Explorar el estado de los datos

- ¿Qué columnas tenemos?
- ¿Cuáles parecen ser clave?
- ¿Hay valores nulos?
- ¿Columnas redundantes o mal nombradas?
- Valores de las columnas 


In [32]:
import pandas as pd
import re
# Cargar datos unificados
df = pd.read_csv('todos_los_establecimientos.csv', encoding='utf-8-sig')

# Dimensiones del dataset
print(f"Filas: {df.shape[0]:,}, Columnas: {df.shape[1]}")

# Ver las primeras columnas y filas
df.head()


Filas: 16,414, Columnas: 17


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,16-01-0026-45,16-031,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO IMPERIAL,5A. CALLE 1-98 ZONA 3,57101061,PATRICIO NAJARRO ASENCIO,MYNOR GUSTAVO IPIÑA ESPAÑA,BASICO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,DOBLE,FIN DE SEMANA,ALTA VERAPAZ
1,16-01-0135-45,16-005,ALTA VERAPAZ,COBAN,INEB ADSCRITO A INSTITUTO 'EMILIO ROSALES PONCE',3A AVE 6-23 ZONA 11,79529782,NORA LILIANA FIGUEROA HERNÁNDEZ,VICTOR HUGO DOMÍNGUEZ REYES,BASICO,OFICIAL,URBANA,ABIERTA,BILINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
2,16-01-0136-45,16-005,ALTA VERAPAZ,COBAN,INEB,6A AVE 1-15 ZONA 4,79513568,NORA LILIANA FIGUEROA HERNÁNDEZ,WUENDY LUCRECIA ESTRADA BEDOYA,BASICO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
3,16-01-0138-45,16-031,ALTA VERAPAZ,COBAN,COLEGIO COBAN,"KM.2 SALIDA A SAN JUAN CHAMELCO, ZONA 8",77945104,PATRICIO NAJARRO ASENCIO,GUSTAVO ADOLFO SIERRA POP,BASICO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
4,16-01-0139-45,16-031,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,77367402,PATRICIO NAJARRO ASENCIO,GILMA DOLORES GUAY PAZ DE LEAL,BASICO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ


In [33]:
df.columns.tolist()


['CODIGO',
 'DISTRITO',
 'DEPARTAMENTO',
 'MUNICIPIO',
 'ESTABLECIMIENTO',
 'DIRECCION',
 'TELEFONO',
 'SUPERVISOR',
 'DIRECTOR',
 'NIVEL',
 'SECTOR',
 'AREA',
 'STATUS',
 'MODALIDAD',
 'JORNADA',
 'PLAN',
 'DEPARTAMENTAL']

In [34]:
df.isna().sum().sort_values(ascending=False)


TELEFONO           424
DIRECTOR            63
DIRECCION           10
SECTOR               0
PLAN                 0
JORNADA              0
MODALIDAD            0
STATUS               0
AREA                 0
CODIGO               0
NIVEL                0
DISTRITO             0
SUPERVISOR           0
ESTABLECIMIENTO      0
MUNICIPIO            0
DEPARTAMENTO         0
DEPARTAMENTAL        0
dtype: int64

La mayoría de los campos están completos y bien distribuidos.

Solo hay algunos casos donde faltan teléfonos y en menor medida nombres de directores o direcciones.


## Paso 1: Normalización de texto

Objetivo: Hacer que todos los textos estén en un formato uniforme, eliminando espacios extra y asegurando que todo esté en mayúsculas para facilitar comparaciones y análisis posteriores.

Columnas a limpiar:
- `ESTABLECIMIENTO`
- `DIRECCION`
- `TELEFONO`
- `SUPERVISOR`
- `DIRECTOR`


In [35]:
# Copia del dataframe original por seguridad
df_limpio = df.copy()

# Lista de columnas de texto a normalizar
columnas_texto = ['ESTABLECIMIENTO', 'DIRECCION', 'TELEFONO', 'SUPERVISOR', 'DIRECTOR']

for col in columnas_texto:
    df_limpio[col] = (
        df_limpio[col]
        .fillna('')                       # Rellena vacíos con cadena vacía
        .astype(str)                      # Asegura que todos sean strings
        .str.replace('\xa0', ' ', regex=False)  # NBSP → espacio
        .apply(lambda x: ' '.join(x.split()))   # Quita espacios extra
        .str.upper()                     # Convierte a mayúsculas
    )
    


## Estado de los datos

- **Datos completos** en la mayoría de las columnas, con valores faltantes solo en:
  - `TELEFONO` (424 casos)
  - `DIRECTOR` (63 casos)
  - `DIRECCION` (10 casos)
- Los campos de texto presentan inconsistencias en el uso de mayúsculas/minúsculas y espacios extra.
- En la columna `TELEFONO` aparecen valores con el formato `79416669.0` debido a que fueron interpretados como números flotantes (`float`) durante la carga del CSV.
- En algunos casos, debemos suponer y tratar que al convertir un número a `float`, se perdió un cero inicial, lo que reduce la longitud del número a 7 dígitos.
- Existen teléfonos vacíos que deben marcarse como `"NO DISPONIBLE"`.
- En general, el formato de los textos no es uniforme (acentos, tildes, caracteres invisibles como `\xa0`).

---

## Operaciones de limpieza a realizar

1. **Normalización de teléfonos**:
   - Quitar el `.0` al final de los valores numéricos.
   - Si el número tiene 7 dígitos, agregar un `0` al inicio para corregir la pérdida del cero inicial.
   - Si la longitud es 0 (vacío), reemplazar por `"NO DISPONIBLE"`.

2. **Estandarización de texto**:
   - Convertir a mayúsculas para evitar diferencias por capitalización.
   - Eliminar espacios al inicio y final, así como espacios repetidos entre palabras.
   - Sustituir caracteres invisibles (como `\xa0`) por espacios.

3. **Unificación de formato**:
   - En campos de texto como `ESTABLECIMIENTO`, `DIRECCION`, `SUPERVISOR` y `DIRECTOR`, aplicar la misma normalización para facilitar la detección de duplicados o errores tipográficos.

4. **Revisión de duplicados**:
   - Detectar registros con nombre de establecimiento y dirección idénticos o muy similares.

---


In [36]:
import re
import pandas as pd

df_limpio = df.copy()

def limpiar_telefono(valor: object) -> str:
    """
    Limpia y normaliza el teléfono:
    - Quita NBSP y espacios extra.
    - Quita sufijo .0 al final.
    - Si tiene 8 dígitos (nacional) => 'XXXX-XXXX'.
    - Si viene con 502/+502 y 8 dígitos => nacional 'XXXX-XXXX'.
    - Si tiene 7 dígitos => NO se completa; se deja tal cual (se marca con bandera).
    - Vacíos => 'NO DISPONIBLE'.
    """
    if pd.isna(valor):
        return "NO DISPONIBLE"

    tel = str(valor).replace('\xa0', ' ').strip()
    tel = re.sub(r'\.0$', '', tel)         # remueve .0 final
    tel = ''.join(tel.split())             # quita espacios internos

    if tel == '':
        return "NO DISPONIBLE"

    solo = re.sub(r'\D', '', tel)          # extrae dígitos

    # +502 / 502
    if solo.startswith('502') and len(solo) == 11:
        solo = solo[3:]                    # conserva los 8 nacionales

    if len(solo) == 8:
        return f"{solo[:4]}-{solo[4:]}"
    elif len(solo) == 7:
        # no completar automáticamente
        return tel

    # Cualquier otro caso: devolver tal cual para inspección posterior
    return tel

# Conservar original y aplicar limpieza
df_limpio['TELEFONO_RAW'] = df_limpio['TELEFONO']
df_limpio['TELEFONO'] = df_limpio['TELEFONO'].apply(limpiar_telefono)

# Banderas útiles
def _flags_tel(t: str) -> pd.Series:
    if t == "NO DISPONIBLE":
        return pd.Series({'telefono_disponible': False, 'telefono_necesita_revision': False})
    d = re.sub(r'\D', '', str(t))
    return pd.Series({
        'telefono_disponible': True,
        'telefono_necesita_revision': (len(d) == 7)
    })

df_limpio[['telefono_disponible','telefono_necesita_revision']] = df_limpio['TELEFONO'].apply(_flags_tel)

# Muestra rápida
df_limpio['TELEFONO'].sample(10)


8484         5632-1089
4269         7882-1508
371      NO DISPONIBLE
13349        4324-2498
11186        3111-2199
5451         3991-4060
14676    NO DISPONIBLE
13962        5012-3760
5359         5517-3413
7140         6631-1112
Name: TELEFONO, dtype: object

---
## Normalizacion de textos:
Evitar espacios dobles, y en los extremos. porque así normalizamos, además también definimos una "AVENIDA" porque evitamos problemas y seguimos 
normalizando
- AV. / AV → AVENIDA

- Z. / ZN → ZONA

---

In [37]:
import re

cols_texto = ['ESTABLECIMIENTO', 'DIRECCION', 'SUPERVISOR', 'DIRECTOR']
cols_texto = [c for c in cols_texto if c in df_limpio.columns]

def normalizar_texto(s: pd.Series) -> pd.Series:
    s = s.fillna('').astype(str)
    s = (
        s.str.replace('\xa0', ' ', regex=False)
         .apply(lambda x: ' '.join(x.split()))
         .str.upper()
    )
    return s

def normalizar_direccion(s: pd.Series) -> pd.Series:
    s = normalizar_texto(s)
    # Reemplazos frecuentes y seguros (ampliados)
    reemplazos = [
        (r'\bAV\.\b', 'AVENIDA'),
        (r'\bAV\b', 'AVENIDA'),
        (r'\bAVE\.?\b', 'AVENIDA'),
        (r'\bAVDA\.?\b', 'AVENIDA'),
        (r'\bBLVD\.?\b', 'BULEVAR'),
        (r'\bCALZ\.?\b', 'CALZADA'),
        (r'\bCARR\.?\b', 'CARRETERA'),
        (r'\bZN\b', 'ZONA'),
        (r'\bZ\.\b', 'ZONA'),
        (r'\bZ\.?\b', 'ZONA'),
        (r'\bKM\.?\b', 'KM'),
        (r'\bNO\.?\s+(\d+)\b', r'NO \1'),  # normaliza "No. 5" / "No 5"
    ]
    for patron, repl in reemplazos:
        s = s.str.replace(patron, repl, regex=True)
    s = s.apply(lambda x: ' '.join(x.split()))
    return s

# Aplicamos y contamos cambios
for col in cols_texto:
    original = df_limpio[col].copy()

    if col == 'DIRECCION':
        df_limpio[col] = normalizar_direccion(df_limpio[col])
    else:
        df_limpio[col] = normalizar_texto(df_limpio[col])

    cambios = (original != df_limpio[col]).sum()
    print(f"Columna '{col}': {cambios} registros modificados.")

df_limpio[cols_texto].head(10)


Columna 'ESTABLECIMIENTO': 0 registros modificados.
Columna 'DIRECCION': 960 registros modificados.
Columna 'SUPERVISOR': 0 registros modificados.
Columna 'DIRECTOR': 63 registros modificados.


,ESTABLECIMIENTO,DIRECCION,SUPERVISOR,DIRECTOR
0,COLEGIO PARTICULAR MIXTO IMPERIAL,5A. CALLE 1-98 ZONA 3,PATRICIO NAJARRO ASENCIO,MYNOR GUSTAVO IPIÑA ESPAÑA
1,INEB ADSCRITO A INSTITUTO 'EMILIO ROSALES PONCE',3A AVENIDA 6-23 ZONA 11,NORA LILIANA FIGUEROA HERNÁNDEZ,VICTOR HUGO DOMÍNGUEZ REYES
2,INEB,6A AVENIDA 1-15 ZONA 4,NORA LILIANA FIGUEROA HERNÁNDEZ,WUENDY LUCRECIA ESTRADA BEDOYA
3,COLEGIO COBAN,"KM2 SALIDA A SAN JUAN CHAMELCO, ZONA 8",PATRICIO NAJARRO ASENCIO,GUSTAVO ADOLFO SIERRA POP
4,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,PATRICIO NAJARRO ASENCIO,GILMA DOLORES GUAY PAZ DE LEAL
5,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,PATRICIO NAJARRO ASENCIO,VIRGINIA SOLANO SERRANO
6,INSTITUTO NACIONAL DE EDUCACION BASICA DE TELE...,ALDEA SAMOX SAN LUCAS,JOSE ARTURO CHOC CHEN,DEBORA ESMERALDA NATARENO FLORES
7,INSTITUTO NACIONAL DE EDUCACION BASICA DE TELE...,ALDEA CAMCAL,JOSE ARTURO CHOC CHEN,DOMINGO TOT COY
8,"LICEO ""MODERNO LATINO""",11 AVENIDA 5-17 ZONA 4,PATRICIO NAJARRO ASENCIO,HÉCTOR ARMANDO TEYUL CHEN
9,COLEGIO PRIVADO MIXTO TECNOLÓGICO EN INFORMÁTICA,"2A. CALLE 12-23, ZONA 4",PATRICIO NAJARRO ASENCIO,JORGE SALVADOR JUÁREZ SIERRA


---
## Tipos de dato seguros:
Para evitar que campos de texto se conviertan en números y pierdan formato.



---

In [38]:
import unicodedata

cols_a_texto = [
    'CODIGO','DISTRITO','DEPARTAMENTO','MUNICIPIO','ESTABLECIMIENTO','DIRECCION',
    'TELEFONO','SUPERVISOR','DIRECTOR','NIVEL','SECTOR','AREA','STATUS',
    'MODALIDAD','JORNADA','PLAN','DEPARTAMENTAL'
]
cols_a_texto = [c for c in cols_a_texto if c in df_limpio.columns]

cambios_tipo = {}
for col in cols_a_texto:
    antes = df_limpio[col].copy()
    df_limpio[col] = (
        df_limpio[col]
          .astype(str)
          .str.replace('\xa0',' ', regex=False)
          .str.strip()
          .str.replace(r'\.0$', '', regex=True)
    )
    cambios_tipo[col] = (antes.astype(str) != df_limpio[col]).sum()

print("Limpieza NROM — Cambios por normalización de tipo:")
for k,v in cambios_tipo.items():
    if v > 0:
        print(f"  {k}: {v} valores ajustados")


Limpieza NROM — Cambios por normalización de tipo:


---
## Estandarización Categórica

Para evitar variantes de escritura (matutina/MATUTINO, vespertino/vespertina, etc.).


---

In [39]:
def normaliza_texto_basico(s: pd.Series) -> pd.Series:
    s = s.fillna('').astype(str)
    s = (
        s.str.replace('\xa0',' ', regex=False)
         .apply(lambda x: ' '.join(x.split()))
         .str.upper()
    )
    return s

# Mapas conservadores (unificados)
map_jornada = {
    'MATUTINO': 'MATUTINA', 'MAT': 'MATUTINA',
    'VESPERTINO': 'VESPERTINA', 'VESP': 'VESPERTINA',
    'NOCTURNO': 'NOCTURNA', 'NOC': 'NOCTURNA',
    'DOBLE JORNADA': 'DOBLE',
    'FINDESEMANA': 'FIN DE SEMANA',
    'FIN DE SEMANA': 'FIN DE SEMANA'
}
map_sector = {
    'PUBLICO': 'OFICIAL', 'PÚBLICO': 'OFICIAL',
    'PRIVADA': 'PRIVADO', 'PÚBLICA': 'OFICIAL', 'PUBLICA': 'OFICIAL',
    'ESTATAL': 'OFICIAL'
}
map_area = {
    'URBANO': 'URBANA',
    'RURAL': 'RURAL'
}
# Unificamos a la taxonomía del dataset (p.ej. ABIERTA/CERRADA/SUSPENDIDA)
map_status = {
    'ACTIVA': 'ABIERTA',
    'INACTIVA': 'CERRADA',
    'SUSPENDIDA': 'SUSPENDIDA'
}

def aplicar_mapa(col: str, mapa: dict):
    if col not in df_limpio.columns:
        return
    original = df_limpio[col].copy()
    df_limpio[col] = normaliza_texto_basico(df_limpio[col]).replace(mapa)
    cambios = (original != df_limpio[col]).sum()
    print(f"Limpieza vals — '{col}': {cambios} valores estandarizados.")

for col, mapa in [('JORNADA', map_jornada),
                  ('SECTOR', map_sector),
                  ('AREA', map_area),
                  ('STATUS', map_status)]:
    aplicar_mapa(col, mapa)

def normalizar_direccion_2(s: pd.Series) -> pd.Series:
    s = normaliza_texto_basico(s)
    reemplazos = [
        (r'\bAV\.\b', 'AVENIDA'),
        (r'\bAV\b', 'AVENIDA'),
        (r'\bAVE\.?\b', 'AVENIDA'),
        (r'\bAVDA\.?\b', 'AVENIDA'),
        (r'\bCALZ\.\b', 'CALZADA'),
        (r'\bCARR\.\b', 'CARRETERA'),
        (r'\bBLVD\.?\b', 'BULEVAR'),
        (r'\bZN\b', 'ZONA'),
        (r'\bZ\.\b', 'ZONA'),
        (r'\bZ\.?\b', 'ZONA'),
        (r'\bKM\.?\b', 'KM')
    ]
    for patron, repl in reemplazos:
        s = s.str.replace(patron, repl, regex=True)
    s = s.apply(lambda x: ' '.join(x.split()))
    return s

if 'DIRECCION' in df_limpio.columns:
    antes = df_limpio['DIRECCION'].copy()
    df_limpio['DIRECCION'] = normalizar_direccion_2(df_limpio['DIRECCION'])
    cambios = (antes != df_limpio['DIRECCION']).sum()
    print(f"Limpieza DIRS — 'DIRECCION': {cambios} valores normalizados.")


Limpieza vals — 'JORNADA': 0 valores estandarizados.
Limpieza vals — 'SECTOR': 0 valores estandarizados.
Limpieza vals — 'AREA': 0 valores estandarizados.
Limpieza vals — 'STATUS': 0 valores estandarizados.
Limpieza DIRS — 'DIRECCION': 0 valores normalizados.


---
## Marca de posibles duplicados

Porque ayudar a detectar registros repetidos sin eliminar filas.



---


In [ ]:

def quitar_acentos(texto: str) -> str:
    texto = unicodedata.normalize('NFD', texto or '')
    return ''.join(ch for ch in texto if unicodedata.category(ch) != 'Mn')

def canonizar_cadena(texto: str) -> str:
    t = (texto or '').upper()
    t = quitar_acentos(t)
    t = re.sub(r'[^A-Z0-9\s]', ' ', t)  # deja letras, números y espacios
    t = ' '.join(t.split())
    return t

# Asegurar tipo texto
for col in ['ESTABLECIMIENTO','MUNICIPIO','DIRECCION']:
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio[col].fillna('').astype(str)

# Llave canónica (fuerte) y suave
if all(c in df_limpio.columns for c in ['ESTABLECIMIENTO','MUNICIPIO','DIRECCION']):
    df_limpio['LLAVE_CANONICA'] = (
        df_limpio['ESTABLECIMIENTO'] + ' | ' +
        df_limpio['MUNICIPIO'] + ' | ' +
        df_limpio['DIRECCION']
    ).apply(canonizar_cadena)

    df_limpio['LLAVE_CANONICA_SUAVE'] = (
        df_limpio['ESTABLECIMIENTO'] + ' | ' +
        df_limpio['MUNICIPIO']
    ).apply(canonizar_cadena)

    conteos = df_limpio['LLAVE_CANONICA'].value_counts(dropna=False)
    df_limpio['POSIBLE_DUPLICADO'] = df_limpio['LLAVE_CANONICA'].map(lambda x: conteos.get(x,0) > 1)

    print("Limpieza DUPS — Posibles duplicados (llave fuerte):",
          int(df_limpio['POSIBLE_DUPLICADO'].sum()))

    # Top 10 llaves con más repeticiones (útil para inspección)
    top10 = conteos[conteos > 1].head(10)
    if not top10.empty:
        print("\nTop 10 llaves canónicas con más repetidos:")
        for k, v in top10.items():
            print(f"  {k}  -> {v} registros")


Limpieza DUPS — Posibles duplicados (llave fuerte): 7571

Top 10 llaves canónicas con más repetidos:
  CENTRO DE ESTUDIOS TECNICOS Y AVANZADOS DE CHIMALTENANGO C E T A CH CHIMALTENANGO 8A AVENIDA 3 59 ZONA 2  -> 12 registros
  INSTITUTO GUILLERMO PUTZEYS ALVAREZ ZONA 1 11 CALLE 3 59  -> 12 registros
  COLEGIO MIXTO PRIVADO SAN JOSE QUETZALTENANGO 20 AVENIDA 1 07 ZONA 1  -> 12 registros
  CENTRO EDUCATIVO MAYA LOS AMATES BARRIO LA CASONA  -> 12 registros
  COLEGIO PRE UNIVERSITARIO FRIEDRICH VON HAYEK QUETZALTENANGO 21 AVENIDA 3 61 ZONA 3  -> 11 registros
  CENTRO EDUCATIVO INTELLECTUS PRE UNIVERSITARIO SAN MIGUEL PETAPA 4TA CALLE 1 70 GRANJA LAS JOYAS ZONA 8 SAN MIGUEL PETAPA  -> 11 registros
  LICEO CANADIENSE SOCIEDAD ANONIMA ZONA 12 DIAGONAL 19 AVENIDA PETAPA 40 54  -> 10 registros
  INSTITUTO IBEROAMERICANO DE ESTUDIOS AVANZADOS DE RETALHULEU RETALHULEU 5A AVENIDA 6 49 ZONA 1  -> 10 registros
  LICEO TECNICO INTEGRAL SANTA TERESA DE JESUS ZONA 1 12 CALLE 10 37  -> 10 registros
  CO

---
## Valores extraños y consistencias 

Para alertar si hay etiquetas fuera de lo común y  para detectar posibles incongruencias municipio–departamento

---

In [41]:
cat_cols = ['JORNADA','SECTOR','AREA','STATUS','MODALIDAD','NIVEL']
cat_cols = [c for c in cat_cols if c in df_limpio.columns]

# Listas conservadoras pero alineadas con los mapas/uso real
esperados = {
    'JORNADA': {'MATUTINA','VESPERTINA','NOCTURNA','DOBLE','FIN DE SEMANA','INTERMEDIA','SIN JORNADA'},
    'SECTOR': {'OFICIAL','PRIVADO','COOPERATIVA','MUNICIPAL'},
    'AREA': {'URBANA','RURAL','SIN ESPECIFICAR'},
    'STATUS': {'ABIERTA','CERRADA','SUSPENDIDA',''},   # permite vacío
    'MODALIDAD': {'MONOLINGUE','BILINGUE'}
    # 'NIVEL': puedes imprimir y revisar; si quieres, agrega un set esperado
}

print("== Revisión de categóricos ==")
for col in cat_cols:
    valores = sorted(df_limpio[col].dropna().unique().tolist())
    print(f"\n{col}: {len(valores)} valores únicos")
    print("Ejemplos:", valores[:15])

    if col in esperados:
        fuera = [v for v in valores if v not in esperados[col]]
        if fuera:
            print(f"  *Valores NO esperados en {col}*:", fuera)
        else:
            print(f"  Todos los valores de {col} están dentro de lo esperado.")


== Revisión de categóricos ==

JORNADA: 6 valores únicos
Ejemplos: ['DOBLE', 'INTERMEDIA', 'MATUTINA', 'NOCTURNA', 'SIN JORNADA', 'VESPERTINA']
  Todos los valores de JORNADA están dentro de lo esperado.

SECTOR: 4 valores únicos
Ejemplos: ['COOPERATIVA', 'MUNICIPAL', 'OFICIAL', 'PRIVADO']
  Todos los valores de SECTOR están dentro de lo esperado.

AREA: 3 valores únicos
Ejemplos: ['RURAL', 'SIN ESPECIFICAR', 'URBANA']
  Todos los valores de AREA están dentro de lo esperado.

STATUS: 1 valores únicos
Ejemplos: ['ABIERTA']
  Todos los valores de STATUS están dentro de lo esperado.

MODALIDAD: 2 valores únicos
Ejemplos: ['BILINGUE', 'MONOLINGUE']
  Todos los valores de MODALIDAD están dentro de lo esperado.

NIVEL: 2 valores únicos
Ejemplos: ['BASICO', 'DIVERSIFICADO']


---
## Columnas vacias y Duplicados exactos

Para ver si alguna columna quedó sin datos y para saber si hay filas idénticas

---

In [ ]:
cols_vacias = []
for col in df_limpio.columns:
    vacia = df_limpio[col].replace('', pd.NA).isna().all()
    if vacia:
        cols_vacias.append(col)

print("\n== Columnas 100% vacías ==")
print(cols_vacias if cols_vacias else "Ninguna columna está completamente vacía.")

# Duplicados exactos (toda la fila)
dup_mask = df_limpio.duplicated(keep=False)
num_dups = int(dup_mask.sum())
print("\n== Duplicados exactos (fila completa) (inicioInciso5) ==")
print(f"Filas que aparecen duplicadas exactamente: {num_dups}")

if num_dups > 0:
    print("Ejemplo de índices duplicados:", df_limpio[dup_mask].index[:10].tolist())

# Extra: top 10 grupos por LLAVE_CANONICA (si existe)
if 'LLAVE_CANONICA' in df_limpio.columns:
    dups_llave = (
        df_limpio[df_limpio['POSIBLE_DUPLICADO']]
        .groupby('LLAVE_CANONICA', dropna=False)
        .size()
        .sort_values(ascending=False)
        .head(10)
    )
    if not dups_llave.empty:
        print("\nTop 10 LLAVE_CANONICA con más repetidos:")
        print(dups_llave)



== Columnas 100% vacías ==
Ninguna columna está completamente vacía.

== Duplicados exactos (fila completa) (inicioInciso5) ==
Filas que aparecen duplicadas exactamente: 0


---
## Duplicidad de establecimiento  

(Inciso 5) 
---